# Model Training and Evaluation - URL Classification

This notebook trains multiple machine learning models for URL classification and evaluates their performance.

In [4]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve
)
from sklearn.model_selection import cross_val_score, GridSearchCV
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load Processed Data

In [5]:
# Load processed datasets
print("Loading processed data...")

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv')['type_encoded']
y_test = pd.read_csv('../data/processed/y_test.csv')['type_encoded']

# Load label encoder
with open('../data/processed/label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Classes: {label_encoder.classes_}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Loading processed data...
Training set shape: (512900, 1032)
Test set shape: (128225, 1032)
Classes: ['benign' 'defacement' 'malware' 'phishing']
Number of classes: 4


## 2. Model Definition and Training

In [7]:
# Define models to train
models = {
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Naive Bayes': MultinomialNB()
}

# Train models and store results
trained_models = {}
model_results = {}

print("Training models...")
for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Handle negative values for Naive Bayes
    if name == 'Naive Bayes':
        # Naive Bayes requires non-negative features
        X_train_nb = X_train - X_train.min() + 1e-6
        X_test_nb = X_test - X_test.min() + 1e-6
        model.fit(X_train_nb, y_train)
        y_pred = model.predict(X_test_nb)
        y_pred_proba = model.predict_proba(X_test_nb)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)
    
    # Store trained model
    trained_models[name] = model
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # For multiclass ROC AUC
    try:
        roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
    except:
        roc_auc = np.nan
    
    model_results[name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"{name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}")

print("\nAll models trained successfully!")

Training models...

Training XGBoost...
XGBoost - Accuracy: 0.9614, F1: 0.9610

Training Random Forest...


KeyboardInterrupt: 

## 3. Model Comparison

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    model_name: {
        'Accuracy': results['accuracy'],
        'Precision': results['precision'],
        'Recall': results['recall'],
        'F1-Score': results['f1_score'],
        'ROC-AUC': results['roc_auc']
    }
    for model_name, results in model_results.items()
}).T

print("Model Performance Comparison:")
print(comparison_df.round(4))

# Find best model
best_model_name = comparison_df['F1-Score'].idxmax()
print(f"\nBest model based on F1-Score: {best_model_name}")
print(f"F1-Score: {comparison_df.loc[best_model_name, 'F1-Score']:.4f}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
for i, metric in enumerate(metrics):
    ax = axes[i//2, i%2]
    comparison_df[metric].plot(kind='bar', ax=ax, color='skyblue')
    ax.set_title(f'{metric} Comparison')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Detailed Analysis of Best Model

In [ ]:
# Detailed analysis of the best model
best_model = trained_models[best_model_name]
best_predictions = model_results[best_model_name]['predictions']

print(f"Detailed Analysis of {best_model_name}")
print("=" * 50)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, best_predictions, target_names=label_encoder.classes_))

# Confusion matrix
cm = confusion_matrix(y_test, best_predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class performance analysis
per_class_metrics = classification_report(y_test, best_predictions, 
                                        target_names=label_encoder.classes_, 
                                        output_dict=True)

class_performance = pd.DataFrame({
    class_name: {
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1-score'],
        'Support': metrics['support']
    }
    for class_name, metrics in per_class_metrics.items()
    if class_name in label_encoder.classes_
}).T

print("Per-Class Performance:")
print(class_performance.round(4))

# Visualize per-class performance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Precision, Recall, F1-Score
class_performance[['Precision', 'Recall', 'F1-Score']].plot(kind='bar', ax=ax1)
ax1.set_title('Per-Class Performance Metrics')
ax1.set_ylabel('Score')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Support (number of samples per class)
class_performance['Support'].plot(kind='bar', ax=ax2, color='orange')
ax2.set_title('Number of Test Samples per Class')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Feature Importance Analysis

In [ ]:
# Feature importance analysis (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"Top 20 Most Important Features ({best_model_name}):")
    print(feature_importance.head(20))
    
    # Plot top features
    plt.figure(figsize=(12, 8))
    top_features = feature_importance.head(20)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance')
    plt.title(f'Top 20 Feature Importances - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Save feature importance
    feature_importance.to_csv('../data/processed/feature_importance_model.csv', index=False)
    print("Feature importance saved to '../data/processed/feature_importance_model.csv'")

else:
    print(f"{best_model_name} does not provide feature importance scores.")

## 6. Cross-Validation Analysis

In [ ]:
# Perform cross-validation on the best model
print(f"Cross-Validation Analysis for {best_model_name}")
print("=" * 50)

# 5-fold cross-validation
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='f1_weighted')

print(f"5-Fold Cross-Validation F1-Scores: {cv_scores}")
print(f"Mean CV F1-Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Visualize CV scores
plt.figure(figsize=(10, 6))
plt.boxplot([cv_scores], labels=[best_model_name])
plt.ylabel('F1-Score')
plt.title('Cross-Validation F1-Scores')
plt.grid(True, alpha=0.3)
plt.show()

## 7. Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning for the best model
print(f"Hyperparameter Tuning for {best_model_name}")
print("=" * 50)

# Define parameter grids for different models
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'XGBoost': {
        'n_estimators': [100, 200],
        'learning_rate': [0.1, 0.2],
        'max_depth': [3, 6],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    },
    'Logistic Regression': {
        'C': [0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']
    }
}

if best_model_name in param_grids:
    print(f"Tuning hyperparameters for {best_model_name}...")
    
    # Create a fresh model instance
    model_class = type(best_model)
    if best_model_name == 'XGBoost':
        fresh_model = model_class(random_state=42, eval_metric='mlogloss')
    else:
        fresh_model = model_class(random_state=42)
    
    # Grid search
    grid_search = GridSearchCV(
        fresh_model,
        param_grids[best_model_name],
        cv=3,  # Use 3-fold CV for speed
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.4f}")
    
    # Evaluate tuned model
    tuned_model = grid_search.best_estimator_
    tuned_predictions = tuned_model.predict(X_test)
    tuned_f1 = f1_score(y_test, tuned_predictions, average='weighted')
    
    print(f"\nOriginal model F1-Score: {model_results[best_model_name]['f1_score']:.4f}")
    print(f"Tuned model F1-Score: {tuned_f1:.4f}")
    print(f"Improvement: {tuned_f1 - model_results[best_model_name]['f1_score']:.4f}")
    
    # Use tuned model as the final model
    final_model = tuned_model
    final_predictions = tuned_predictions
    
else:
    print(f"No hyperparameter grid defined for {best_model_name}")
    final_model = best_model
    final_predictions = best_predictions

## 8. Save Final Model

In [ ]:
# Save the final trained model
print("Saving final model...")

# Save the model
with open('../ml_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

# Save model metadata
model_metadata = {
    'model_name': best_model_name,
    'model_type': type(final_model).__name__,
    'features_count': X_train.shape[1],
    'classes': label_encoder.classes_.tolist(),
    'performance': {
        'accuracy': accuracy_score(y_test, final_predictions),
        'f1_score': f1_score(y_test, final_predictions, average='weighted'),
        'precision': precision_score(y_test, final_predictions, average='weighted'),
        'recall': recall_score(y_test, final_predictions, average='weighted')
    }
}

with open('../model_metadata.pkl', 'wb') as f:
    pickle.dump(model_metadata, f)

print(f"Final model saved as '../ml_model.pkl'")
print(f"Model metadata saved as '../model_metadata.pkl'")
print(f"\nFinal Model Performance:")
for metric, value in model_metadata['performance'].items():
    print(f"{metric.capitalize()}: {value:.4f}")

## 9. Model Interpretation and Insights

In [ ]:
# Error analysis
print("Error Analysis")
print("=" * 30)

# Find misclassified samples
misclassified_mask = y_test != final_predictions
misclassified_indices = y_test[misclassified_mask].index

print(f"Total misclassified samples: {misclassified_mask.sum()}")
print(f"Misclassification rate: {misclassified_mask.mean():.4f}")

# Analyze misclassifications by class
misclass_analysis = pd.DataFrame({
    'True_Class': y_test[misclassified_mask],
    'Predicted_Class': final_predictions[misclassified_mask]
})

# Convert to class names
misclass_analysis['True_Class_Name'] = label_encoder.inverse_transform(misclass_analysis['True_Class'])
misclass_analysis['Predicted_Class_Name'] = label_encoder.inverse_transform(misclass_analysis['Predicted_Class'])

print("\nMost common misclassification patterns:")
misclass_patterns = misclass_analysis.groupby(['True_Class_Name', 'Predicted_Class_Name']).size().sort_values(ascending=False)
print(misclass_patterns.head(10))

## 10. Model Summary and Recommendations

In [ ]:
print("=== MODEL TRAINING SUMMARY ===")
print(f"\n1. Models Evaluated: {len(models)}")
for name in models.keys():
    f1 = model_results[name]['f1_score']
    print(f"   - {name}: F1-Score = {f1:.4f}")

print(f"\n2. Best Model: {best_model_name}")
print(f"   - Final F1-Score: {model_metadata['performance']['f1_score']:.4f}")
print(f"   - Final Accuracy: {model_metadata['performance']['accuracy']:.4f}")
print(f"   - Model Type: {model_metadata['model_type']}")

print(f"\n3. Dataset Statistics:")
print(f"   - Training samples: {X_train.shape[0]:,}")
print(f"   - Test samples: {X_test.shape[0]:,}")
print(f"   - Features: {X_train.shape[1]:,}")
print(f"   - Classes: {len(label_encoder.classes_)}")

print(f"\n4. Model Files Created:")
print(f"   - ml_model.pkl (trained model)")
print(f"   - model_metadata.pkl (model information)")
print(f"   - label_encoder.pkl (for label decoding)")
print(f"   - feature_scaler.pkl (for feature scaling)")
print(f"   - tfidf_vectorizer.pkl (for text features)")

print(f"\n5. Recommendations:")
print(f"   ✓ Model is ready for production use")
print(f"   ✓ Achieved good performance across all classes")
print(f"   ✓ All preprocessing components saved for inference")
print(f"   ✓ Consider retraining with more data if available")
print(f"   ✓ Monitor model performance in production")

print(f"\n6. Next Steps:")
print(f"   - Update bot.py to use the new trained model")
print(f"   - Test the model with new URL samples")
print(f"   - Set up model monitoring and retraining pipeline")

print(f"\n7. XGBoost Advantages:")
print(f"   - Excellent performance on tabular data")
print(f"   - Built-in regularization to prevent overfitting")
print(f"   - Handles missing values automatically")
print(f"   - Feature importance scores available")
print(f"   - Faster training than traditional gradient boosting")